# 03: 文本分块深度剖析

## 概述

文本分块（Text Splitting / Chunking）是 RAG（检索增强生成）系统中最关键的预处理步骤之一。分块策略直接影响检索质量：

- **块太大**：检索精度下降，包含过多噪声信息，浪费LLM上下文窗口
- **块太小**：丢失上下文，无法捕获完整的语义信息
- **边界不当**：将相关句子从中切断，导致语义碎片化

本 Notebook 深入实现并比较 **5种主流分块策略**：

| # | 策略 | 核心思想 | 适用场景 |
|---|------|---------|--------|
| 1 | **Fixed-Length** | 按固定字符/Token数切分 | 简单场景、快速原型 |
| 2 | **Recursive Character** | 按分隔符层级递归切分 | 通用文档、代码 |
| 3 | **Sentence-Aware** | 基于句子边界切分 | 自然语言文档 |
| 4 | **Semantic** | 基于语义相似度切分 | 高质量语义检索 |
| 5 | **Adaptive** | 自适应文档类型选择策略 | 多类型文档混合场景 |

我们将用一篇中文技术长文作为测试文档，逐一实现、演示并对比这5种策略的实际效果。

## 1. 环境准备

安装所需的依赖库并导入模块。

In [ ]:
# !pip install numpy tiktoken spacy matplotlib seaborn scikit-learn
# !python -m spacy download zh_core_web_sm
# !python -m spacy download en_core_web_sm

import re
import math
import warnings
from typing import List, Tuple, Dict, Optional, Callable
from dataclasses import dataclass, field
from collections import Counter

import numpy as np

warnings.filterwarnings('ignore')

# ---------- tiktoken ----------
try:
    import tiktoken
    TIKTOKEN_AVAILABLE = True
    ENCODING = tiktoken.get_encoding("cl100k_base")
except Exception:
    TIKTOKEN_AVAILABLE = False
    ENCODING = None
    print("[WARN] tiktoken not available; falling back to char-length.")

# ---------- spaCy ----------
try:
    import spacy
    nlp_zh = spacy.load("zh_core_web_sm")
    nlp_en = spacy.load("en_core_web_sm")
    SPACY_AVAILABLE = True
    print("[OK] spaCy models loaded (zh + en).")
except Exception:
    SPACY_AVAILABLE = False
    nlp_zh = None
    nlp_en = None
    print("[WARN] spaCy models not available; using regex fallback.")

# ---------- sklearn / TfidfVectorizer for semantic fallback ----------
try:
    from sklearn.feature_extraction.text import TfidfVectorizer
    from sklearn.metrics.pairwise import cosine_similarity as sklearn_cosine
    SKLEARN_AVAILABLE = True
except Exception:
    SKLEARN_AVAILABLE = False
    print("[WARN] scikit-learn not available; using word-overlap similarity.")

# ---------- matplotlib / seaborn ----------
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import seaborn as sns

sns.set_style("whitegrid")
plt.rcParams['font.sans-serif'] = ['SimHei', 'Microsoft YaHei', 'DejaVu Sans']
plt.rcParams['axes.unicode_minus'] = False

print("\n=== 环境准备完成 ===")
print(f"tiktoken: {TIKTOKEN_AVAILABLE}")
print(f"spaCy:    {SPACY_AVAILABLE}")
print(f"sklearn:  {SKLEARN_AVAILABLE}")

## 2. 准备测试文档

我们创建一篇 3000+ 字的中文技术文章，涵盖机器学习的历史、分支技术和未来展望。这篇文章将作为所有分块策略的测试输入。

In [ ]:
test_text = """
机器学习：从历史到未来的全面剖析

第一章 机器学习的历史演进

机器学习作为人工智能的一个重要分支，其发展历程跨越了七十余年。早在20世纪50年代，图灵提出了“机器能思考吗”这一著名问题，为机器学习奠定了哲学基础。1952年，Arthur Samuel在IBM开发了首个跳棋学习程序，这是机器学习领域最早的实践之一。1957年，Frank Rosenblatt提出了感知机模型，这是第一个能够从经验中学习的神经网络。然而，1969年Marvin Minsky和Seymour Papert在《感知机》一书中指出了单层感知机的局限性，导致神经网络研究进入了长达十余年的“AI寒冬”。

到了20世纪80年代，机器学习迎来了新的春天。1986年，Rumelhart、Hinton和Williams提出了反向传播算法，解决了多层神经网络的训练问题。与此同时，决策树算法如ID3和C4.5也被广泛采用，支持向量机SVM在90年代成为分类任务的主流方法。1997年，IBM的深蓝计算机击败了国际象棋世界冠军Kasparov，展示了机器学习的巨大潜力。

2012年是深度学习复兴的标志性年份。Alex Krizhevsky等人提出的AlexNet在ImageNet图像分类竞赛中大幅刷新纪录，证明了深度卷积神经网络在大规模数据上的卓越性能。此后，VGGNet、ResNet、Inception等架构相继涌现，深度学习在计算机视觉领域取得了突破性进展。

第二章 监督学习：从标注数据中学习

监督学习是机器学习最经典的分支。它的核心思想是从带有标签的训练数据中学习一个映射函数f: X -> Y，使得模型能够对未见过的输入做出准确预测。监督学习主要包括分类任务和回归任务两大类。

在分类任务中，模型需要将输入分配到预定义的类别中。常见的分类算法包括逻辑回归、K近邻KNN、支持向量机SVM、决策树、随机森林和梯度提升树如XGBoost、LightGBM。这些算法各有优势：逻辑回归简单且可解释性强，SVM在高维空间中表现出色，而树模型能够捕捉复杂的非线性关系。随机森林通过集成多棵决策树来降低过拟合风险，XGBoost则在工业界被广泛使用，凭借其高效和准确性赢得了众多Kaggle竞赛。

回归任务则预测连续值。线性回归是最基础的回归方法，但现实世界的问题往往更加复杂。岭回归和Lasso回归通过正则化项来控制模型复杂度。多项式回归可以拟合非线性曲线。在工程实践中，回归模型被广泛应用于房价预测、股票走势分析、气温预测等领域。

第三章 无监督学习：发现数据中的隐藏结构

无监督学习处理的是没有标签的数据。目标是在数据中寻找隐藏的模式、结构或关系。这是更具挑战性但也更贴近实际问题的场景，因为大多数现实世界的数据都没有标注。

聚类是无监督学习中最常见的任务。K-means算法通过迭代优化簇中心来划分数据，简单高效但需要预先指定簇的数量K。DBSCAN基于密度进行聚类，能够发现任意形状的簇并自动识别噪声点。层次聚类则构建树状的聚类结构，允许在不同粒度上分析数据。高斯混合模型GMM提供了概率框架下的软聚类能力，每个数据点以概率形式归属于各个簇。

降维是无监督学习的另一个重要方向。主成分分析PCA通过线性变换将高维数据投影到低维空间，同时保留尽可能多的方差。t-SNE和UMAP是非线性降维技术，特别适合高维数据的可视化。自编码器Autoencoder使用神经网络学习数据的压缩表示，在去噪和异常检测中表现优异。

第四章 深度学习：神经网络的复兴

深度学习是机器学习中增长最快的子领域。它使用多层人工神经网络来学习数据的层次化表示。低级层检测简单特征如边缘和纹理，高级层组合这些特征形成更复杂的模式如物体部件和整体结构。

卷积神经网络CNN在计算机视觉领域占据主导地位。ResNet引入的残差连接使得训练超过100层的网络成为可能。目标检测框架如Faster R-CNN和YOLO实现了实时物体检测。语义分割网络如U-Net在医学影像分析中挽救了无数生命。

循环神经网络RNN及其变体LSTM和GRU专门处理序列数据，如文本、语音和时间序列。然而，RNN的训练效率较低且难以并行化。注意力机制的提出彻底改变了序列建模的范式。

第五章 Transformer架构：注意力就是一切

2017年，Google团队在论文Attention Is All You Need中提出了Transformer架构，这可能是机器学习领域近十年最重要的突破。Transformer完全基于自注意力机制，摒弃了循环和卷积结构。其核心创新包括多头注意力Multi-Head Attention、位置编码Positional Encoding和层归一化Layer Normalization。

BERT由Google在2018年发布，使用了Transformer的编码器部分。它通过掩码语言模型MLM和下一句预测NSP两个预训练任务，在11项自然语言处理基准测试上取得了最佳成绩。BERT的出现标志着NLP领域进入“预训练+微调”时代。

GPT系列由OpenAI开发，使用了Transformer的解码器部分。从GPT-1到GPT-4，模型规模和能力呈指数级增长。GPT-3拥有1750亿参数，展示了令人惊叹的少样本学习能力。GPT-4进一步提升了推理和多模态能力。ChatGPT的发布引爆了全球AI应用热潮。

第六章 大语言模型与涌现能力

大语言模型LLM是指参数规模达到数十亿甚至数万亿的语言模型。当模型规模超过某个临界点时，会表现出“涌现能力”Emergent Abilities——这些能力在较小模型中没有被明确训练，却在大模型中自然出现。涌现能力包括上下文学习In-Context Learning、思维链推理Chain-of-Thought、代码生成和数学推理等。

LLM的训练流程通常包括三个阶段：预训练在大规模互联网文本上进行自监督学习，监督微调使用人工标注的高质量指令数据进行训练，以及基于人类反馈的强化学习RLHF来对齐人类偏好。这一流程使得LLM能够理解复杂指令并生成有用、安全的回复。

然而，LLM也面临重大挑战。幻觉问题Hallucination使得模型生成看似合理但事实错误的内容。知识截止日期限制了模型对最新信息的掌握。推理计算成本高昂，部署大规模模型需要巨大的资金投入。

第七章 RAG技术：检索增强生成

检索增强生成RAG是解决LLM局限性的关键技术。RAG的核心思想是在生成答案之前，先从外部知识库中检索相关文档，然后将检索结果作为上下文提供给LLM进行生成。这相当于给LLM配备了一个可以随时查阅的外部记忆。

一个典型的RAG系统包括以下几个核心组件：文档加载器负责从各种格式PDF、网页、Word中提取文本；文本分块器将长文档切分为适当大小的块；嵌入模型将文本块转换为向量表示；向量数据库存储和索引这些向量；检索器根据用户查询找到最相关的文档块；最后，生成器基于检索到的上下文和用户查询生成最终答案。

文本分块是RAG流程中至关重要但常被忽视的环节。分块的大小、重叠和边界策略直接影响检索质量。研究表明，对于中文技术文档，推荐的分块大小为400到800个字符，重叠为分块大小的10%到20%。语义感知分块相比固定长度分块，在检索准确率上能提升15%到25%。

高级RAG技术包括查询重写Query Rewriting、混合检索Hybrid Search结合稀疏和稠密检索、重排序Reranking对初检结果进行精排，以及自反思Self-Reflection让LLM评估和修正自己的生成结果。这些技术的组合使用能够显著提升RAG系统的整体表现。

第八章 未来展望：走向通用人工智能

机器学习的未来充满无限可能。多模态模型正在打破文本、图像、音频和视频之间的界限。GPT-4V和Gemini等模型已经展示了同时理解和生成多种模态内容的能力。具身智能Embodied AI将机器学习与机器人技术结合，让智能体能够在物理世界中感知和行动。

AI安全和对齐Alignment是确保AI系统造福人类的关键研究方向。随着AI能力不断增强，我们需要确保AI的目标和价值观与人类保持一致。可解释性研究试图打开神经网络的“黑箱”，让人类理解AI的决策过程。高效的神经网络架构如状态空间模型SSM正在挑战Transformer的主导地位，在长序列建模任务中展现出了巨大的潜力。

展望未来，机器学习将继续深刻改变科学发现、医疗健康、教育培训和创意产业。从蛋白质结构预测到个性化医疗，从自动驾驶到智能助手，机器学习的应用正在渗透到社会的每个角落。我们正站在一个新时代的门槛上，机器学习的潜力才刚刚开始被释放。

结论：机器学习经过七十多年的发展，从简单的感知机演变为拥有数千亿参数的深度神经网络。监督学习和无监督学习构成了其方法论基础，深度学习和Transformer架构推动了近十年的技术革命，大语言模型展示了令人惊叹的涌现能力，而RAG技术为LLM应用提供了坚实的基础。未来，走向通用人工智能的道路虽然漫长，但机器学习的持续进步让我们对未来充满信心。
"""

print(f"测试文档总字符数: {len(test_text)}")
print(f"测试文档估计Token数 (cl100k_base中文约2字符/token): ~{len(test_text)//2}")
print(f"\n--- 文档前200字预览 ---\n{test_text[:200]}...")

## 3. 策略1: Fixed-Length 分块

**核心思想**：按固定长度切分文本，不关心语义边界。实现最简单，性能最高。

**特点**：
- 每个块大小一致，便于批量处理和向量化
- 可以基于字符数或 Token 数
- 使用 overlap 保持块之间的连续性
- **缺点**：可能在句子中间切断，导致语义碎片化

In [ ]:
@dataclass
class FixedLengthSplitter:
    """
    Fixed-Length text chunker.

    Supports two modes:
      - char-based : chunk_size in characters
      - tiktoken-based : chunk_size in tokens (cl100k_base)
    """
    chunk_size: int = 500
    overlap: int = 50
    use_tiktoken: bool = False  # True -> token-based, False -> char-based

    def split(self, text: str) -> List[str]:
        if self.use_tiktoken and TIKTOKEN_AVAILABLE and ENCODING is not None:
            return self._split_by_tokens(text)
        else:
            return self._split_by_chars(text)

    # ------------------------------------------------------------------
    def _split_by_chars(self, text: str) -> List[str]:
        chunks = []
        start = 0
        text_len = len(text)
        while start < text_len:
            end = min(start + self.chunk_size, text_len)
            chunks.append(text[start:end])
            start += (self.chunk_size - self.overlap)
        return chunks

    # ------------------------------------------------------------------
    def _split_by_tokens(self, text: str) -> List[str]:
        tokens = ENCODING.encode(text)
        chunks = []
        start = 0
        total = len(tokens)
        step = max(1, self.chunk_size - self.overlap)
        while start < total:
            end = min(start + self.chunk_size, total)
            chunk_tokens = tokens[start:end]
            chunks.append(ENCODING.decode(chunk_tokens))
            start += step
        return chunks


print("[OK] FixedLengthSplitter defined.")

In [ ]:
# 使用 FixedLengthSplitter
fixed_splitter = FixedLengthSplitter(chunk_size=500, overlap=50)
fixed_chunks = fixed_splitter.split(test_text)

print(f"策略1 Fixed-Length: 共 {len(fixed_chunks)} 个chunk\n")
for i, ch in enumerate(fixed_chunks[:3]):
    print(f"--- Chunk {i+1} (长度={len(ch)}字符) ---")
    print(ch[:150] + "..." if len(ch) > 150 else ch)
    print()

# 如果 tiktoken 可用，也展示 token-based 版本
if TIKTOKEN_AVAILABLE:
    token_splitter = FixedLengthSplitter(chunk_size=200, overlap=20, use_tiktoken=True)
    token_chunks = token_splitter.split(test_text)
    print(f"[Token版] 共 {len(token_chunks)} 个chunk")
    print(f"[Token版] Chunk 0 长度: {len(ENCODING.encode(token_chunks[0]))} tokens / {len(token_chunks[0])} chars")

## 4. 策略2: Recursive Character 分块

**核心思想**：使用分隔符层级递归切分。先尝试用较大的分隔符（段落），如果还是太长，再尝试小分隔符（句子、分句、逗号、空格）。

**分隔符层级（中文优先）**：
1. 双换行 `\n\n` - 段落边界
2. 单换行 `\n` - 手动换行
3. 句号 `。` - 中文句子结束
4. 中文分号 `；`
5. 英文分号 `;`
6. 中文逗号 `，`
7. 英文逗号 `,`
8. 空格 ` `
9. 空字符串 `""` - 逐字符切分（最终兜底）

这是 LangChain 的 RecursiveCharacterTextSplitter 的核心逻辑。

In [ ]:
@dataclass
class RecursiveCharacterSplitter:
    """
    Recursively split text by a hierarchy of separators.

    Tries the coarsest separator first; if a chunk is still too large
    it falls back to finer separators.
    """
    chunk_size: int = 500
    chunk_overlap: int = 50
    separators: List[str] = field(default_factory=lambda: [
        "\n\n", "\n", "。", "；", ";", "，", ",", " ", ""
    ])

    def split(self, text: str) -> List[str]:
        return self._split_text(text, self.separators)

    def _split_text(self, text: str, separators: List[str]) -> List[str]:
        final_chunks = []
        # 取第一个（最粗的）分隔符
        separator = separators[-1] if not separators else separators[0]
        next_separators = separators[1:] if len(separators) > 1 else []

        if not separator:
            # 兜底：逐字符切
            return self._merge_splits(list(text), next_separators)

        splits = text.split(separator)
        return self._merge_splits(splits, next_separators, separator)

    def _merge_splits(
        self,
        splits: List[str],
        next_separators: List[str],
        separator: str = ""
    ) -> List[str]:
        """Merge small splits and re-split oversized ones."""
        chunks = []
        current_chunk = ""
        sep_len = len(separator)

        for s in splits:
            test_chunk = current_chunk + (separator if current_chunk else "") + s

            if len(test_chunk) <= self.chunk_size:
                current_chunk = test_chunk
            else:
                # 保存当前块
                if current_chunk:
                    chunks.append(current_chunk)

                # 当前分裂片段太大，尝试更细的分隔符
                if next_separators:
                    sub_chunks = self._split_text(s, next_separators)
                    # 应用 overlap: 在前一个子块和后续之间添加重叠
                    if chunks and self.chunk_overlap > 0:
                        prev = chunks[-1]
                        if len(prev) > self.chunk_overlap:
                            sub_chunks[0] = prev[-self.chunk_overlap:] + sub_chunks[0]
                    chunks.extend(sub_chunks)
                    current_chunk = sub_chunks[-1] if sub_chunks else ""
                else:
                    # 无更多分隔符：逐字符切
                    if len(s) <= self.chunk_size:
                        current_chunk = s
                    else:
                        char_chunks = self._split_text(s, [])
                        chunks.extend(char_chunks)
                        current_chunk = char_chunks[-1] if char_chunks else ""

        if current_chunk:
            chunks.append(current_chunk)

        return chunks


print("[OK] RecursiveCharacterSplitter defined.")

In [ ]:
# 使用 RecursiveCharacterSplitter
rec_splitter = RecursiveCharacterSplitter(chunk_size=500, chunk_overlap=50)
rec_chunks = rec_splitter.split(test_text)

print(f"策略2 Recursive Character: 共 {len(rec_chunks)} 个chunk\n")
for i, ch in enumerate(rec_chunks[:3]):
    print(f"--- Chunk {i+1} (长度={len(ch)}字符) ---")
    print(ch[:200] + "..." if len(ch) > 200 else ch)
    print()

# 对比 Fixed-Length
print("--- 对比 Fixed-Length vs Recursive ---")
print(f"Fixed-Length chunk 1 的结尾字符: ...{fixed_chunks[0][-30:]}")
print(f"Recursive   chunk 1 的结尾字符: ...{rec_chunks[0][-30:]}")
print("\n观察：Recursive 的边界更可能落在句号或换行等自然分隔符处。")

## 5. 策略3: Sentence-Aware 分块

**核心思想**：先进行句子分割，确保每个句子保持完整，然后将句子合并成块直到接近 chunk_size。

**关键保证**：不将任何句子从中切断。

**实现方式**：
1. 优先使用 spaCy 的句子边界检测（支持中文和英文）
2. 当 spaCy 不可用时，回退到正则表达式

In [ ]:
@dataclass
class SentenceSplitter:
    """
    Sentence-aware chunker.

    Uses spaCy for sentence boundary detection with regex fallback.
    Merges sentences until chunk_size is reached.
    """
    chunk_size: int = 500
    chunk_overlap: int = 50
    language: str = "zh"  # "zh" or "en"

    # 中文句子结束标记的正则
    _ZH_SENT_PATTERN: str = r'(?<=[。！？；\n])(?=[^。！？；\n])'
    # 英文句子结束标记的正则
    _EN_SENT_PATTERN: str = r'(?<=[.!?;\n])\s+(?=[A-Z])'

    def _split_sentences(self, text: str) -> List[str]:
        """Split text into sentences."""
        if SPACY_AVAILABLE and self.language == "zh" and nlp_zh is not None:
            doc = nlp_zh(text)
            return [sent.text for sent in doc.sents]
        elif SPACY_AVAILABLE and self.language == "en" and nlp_en is not None:
            doc = nlp_en(text)
            return [sent.text for sent in doc.sents]
        else:
            return self._regex_split_sentences(text)

    def _regex_split_sentences(self, text: str) -> List[str]:
        """Regex-based sentence splitter (fallback)."""
        # 先按中文标点切分
        pattern = r'(?<=[。！？；\n])'
        parts = re.split(pattern, text)
        # 合并过短的片段
        sentences = []
        buf = ""
        for p in parts:
            if not p.strip():
                if buf:
                    buf += p
                continue
            buf += p
            if len(buf.strip()) >= 5:  # 至少5字符才算一个句子
                sentences.append(buf)
                buf = ""
        if buf.strip():
            sentences.append(buf)
        return sentences

    def split(self, text: str) -> List[str]:
        sentences = self._split_sentences(text)
        chunks = []
        i = 0
        n = len(sentences)
        while i < n:
            chunk = sentences[i]
            j = i + 1
            while j < n and len(chunk) + len(sentences[j]) <= self.chunk_size:
                chunk += sentences[j]
                j += 1
            chunks.append(chunk)

            # 前进到下一个起始句；如有重叠则回退若干句
            if self.chunk_overlap > 0 and j > i + 1:
                overlap_chars = 0
                next_i = j
                for k in range(j - 1, i, -1):
                    overlap_chars += len(sentences[k])
                    if overlap_chars >= self.chunk_overlap:
                        next_i = k
                        break
                # 确保至少前进一个句子
                i = max(i + 1, next_i)
            else:
                i = j

        return chunks


print("[OK] SentenceSplitter defined.")

In [ ]:
# 使用 SentenceSplitter
sent_splitter = SentenceSplitter(chunk_size=500, chunk_overlap=50, language="zh")
sent_chunks = sent_splitter.split(test_text)

print(f"策略3 Sentence-Aware: 共 {len(sent_chunks)} 个chunk\n")
for i, ch in enumerate(sent_chunks[:3]):
    print(f"--- Chunk {i+1} (长度={len(ch)}字符) ---")
    print(ch[:200] + "..." if len(ch) > 200 else ch)
    # 检查结尾是否是句子结束标记
    last_char = ch.strip()[-1] if ch.strip() else ""
    is_sent_end = last_char in "。！？"
    print(f"  -> 结尾字符: '{last_char}' | 是句子结尾: {is_sent_end}")
    print()

# 统计句子完整性
complete_sentence_ends = sum(
    1 for ch in sent_chunks if ch.strip() and ch.strip()[-1] in "。！？"
)
print(f"句子完整结束的chunk数: {complete_sentence_ends}/{len(sent_chunks)}")
print("\n对比：Fixed-Length 可能在任意位置切断，Sentence-Aware 确保在句子边界。")

## 6. 策略4: Semantic 分块

**核心思想**：基于语义相似度决定分块边界。计算相邻句子之间的语义相似度，当相似度出现"低谷"（dip）时，意味着主题发生了变化，应该在此处分块。

**实现步骤**：
1. 将文档拆分为句子
2. 获取每个句子的嵌入向量
3. 计算相邻句子之间的余弦相似度
4. 找到相似度低于阈值的位置作为分块边界
5. 合并相似度高于阈值的相邻句子

**嵌入方案（按优先级）**：
1. OpenAI text-embedding-3-small（需 API key）
2. sentence-transformers 本地模型
3. TF-IDF 向量化（sklearn）
4. 简单词重叠相似度（纯 Python）

In [ ]:
@dataclass
class SemanticSplitter:
    """
    Semantic chunker using sentence embeddings and cosine similarity.

    Splits at "dips" in similarity between adjacent sentences.
    """
    chunk_size: int = 500
    chunk_overlap: int = 50
    similarity_threshold: float = 0.6
    embedding_method: str = "auto"  # "openai" | "sentence-transformers" | "tfidf" | "word-overlap" | "auto"

    def __post_init__(self):
        self._scores: List[float] = []  # 存储相邻句子相似度用于可视化

    def _get_sentences(self, text: str) -> List[str]:
        """Use regex to split into sentences."""
        pattern = r'(?<=[。！？；\n])'
        parts = re.split(pattern, text)
        sentences = []
        buf = ""
        for p in parts:
            buf += p
            stripped = buf.strip()
            if len(stripped) >= 10:
                sentences.append(stripped)
                buf = ""
        if buf.strip():
            sentences.append(buf.strip())
        return sentences

    def _compute_similarities(self, sentences: List[str]) -> np.ndarray:
        """
        Compute pairwise cosine similarities between adjacent sentences.
        Returns array of length len(sentences)-1.
        """
        if len(sentences) < 2:
            return np.array([])

        embeddings = self._get_embeddings(sentences)
        sims = []
        for i in range(len(embeddings) - 1):
            a, b = embeddings[i], embeddings[i + 1]
            dot = np.dot(a, b)
            norm_a = np.linalg.norm(a)
            norm_b = np.linalg.norm(b)
            if norm_a == 0 or norm_b == 0:
                sims.append(0.0)
            else:
                sims.append(float(dot / (norm_a * norm_b)))
        return np.array(sims)

    def _get_embeddings(self, sentences: List[str]) -> List[np.ndarray]:
        """
        Get embeddings using the best available method.
        Priority: openai > sentence-transformers > sklearn-tfidf > word-overlap
        """
        method = self.embedding_method
        if method == "auto":
            # 自动选择最佳可用方法
            if SKLEARN_AVAILABLE:
                method = "tfidf"
            else:
                method = "word-overlap"

        if method == "openai":
            return self._embed_openai(sentences)
        elif method == "sentence-transformers":
            return self._embed_sentence_transformers(sentences)
        elif method == "tfidf":
            return self._embed_tfidf(sentences)
        else:
            return self._embed_word_overlap(sentences)

    def _embed_openai(self, sentences: List[str]) -> List[np.ndarray]:
        """Embed using OpenAI API."""
        try:
            from openai import OpenAI
            import os
            client = OpenAI(api_key=os.environ.get("OPENAI_API_KEY", ""))
            resp = client.embeddings.create(
                model="text-embedding-3-small",
                input=sentences
            )
            return [np.array(d.embedding) for d in resp.data]
        except Exception:
            # 降级
            if SKLEARN_AVAILABLE:
                return self._embed_tfidf(sentences)
            return self._embed_word_overlap(sentences)

    def _embed_sentence_transformers(self, sentences: List[str]) -> List[np.ndarray]:
        """Embed using sentence-transformers."""
        try:
            from sentence_transformers import SentenceTransformer
            model = SentenceTransformer("paraphrase-multilingual-MiniLM-L12-v2")
            vecs = model.encode(sentences)
            return [np.array(v) for v in vecs]
        except Exception:
            if SKLEARN_AVAILABLE:
                return self._embed_tfidf(sentences)
            return self._embed_word_overlap(sentences)

    def _embed_tfidf(self, sentences: List[str]) -> List[np.ndarray]:
        """Embed using sklearn TfidfVectorizer."""
        vectorizer = TfidfVectorizer(
            analyzer='char_wb',
            ngram_range=(2, 4),
            max_features=256
        )
        tfidf_matrix = vectorizer.fit_transform(sentences)
        return [np.array(tfidf_matrix[i].toarray()).flatten() for i in range(len(sentences))]

    def _embed_word_overlap(self, sentences: List[str]) -> List[np.ndarray]:
        """Simple word-overlap embedding: binary vector of shared vocabulary."""
        # 构建全局词汇表
        all_words = set()
        tokenized = []
        for s in sentences:
            # 中文按字符+词简单处理
            words = set(re.findall(r'[\u4e00-\u9fff]+|[a-zA-Z]+', s.lower()))
            tokenized.append(words)
            all_words.update(words)
        word_list = list(all_words)[:500]  # 限制维度
        word_to_idx = {w: i for i, w in enumerate(word_list)}
        embeddings = []
        for words in tokenized:
            vec = np.zeros(len(word_list))
            for w in words:
                if w in word_to_idx:
                    vec[word_to_idx[w]] = 1.0
            embeddings.append(vec)
        return embeddings

    def split(self, text: str) -> List[str]:
        sentences = self._get_sentences(text)
        if len(sentences) == 0:
            return []
        if len(sentences) == 1:
            return sentences

        sims = self._compute_similarities(sentences)
        self._scores = sims.tolist()

        # 确定分割点：相似度低于阈值的位置
        break_indices = [
            i for i, s in enumerate(sims) if s < self.similarity_threshold
        ]

        # 从分割点构建chunks
        chunks = []
        start = 0
        for bp in break_indices:
            end = bp + 1  # bp 是 sims[i] 的索引，对应 sentences[i] 和 sentences[i+1] 之间
            segment = "".join(sentences[start:end])
            if len(segment) > self.chunk_size * 1.5:
                # 过大的段，进一步拆分
                sub_parts = self._split_long_segment(sentences[start:end])
                chunks.extend(sub_parts)
            else:
                chunks.append(segment)
            start = end

        # 最后一段
        if start < len(sentences):
            final_segment = "".join(sentences[start:])
            if len(final_segment) > self.chunk_size * 1.5:
                chunks.extend(self._split_long_segment(sentences[start:]))
            else:
                chunks.append(final_segment)

        return [c for c in chunks if c.strip()]

    def _split_long_segment(self, sentences: List[str]) -> List[str]:
        """Fallback: split long segment by char count."""
        text = "".join(sentences)
        chunks = []
        for i in range(0, len(text), self.chunk_size):
            chunks.append(text[i:i + self.chunk_size])
        return chunks

    @property
    def scores(self) -> List[float]:
        return self._scores


print("[OK] SemanticSplitter defined.")

In [ ]:
# 使用 SemanticSplitter
sem_splitter = SemanticSplitter(
    chunk_size=500,
    chunk_overlap=50,
    similarity_threshold=0.3,
    embedding_method="auto"
)
sem_chunks = sem_splitter.split(test_text)

print(f"策略4 Semantic: 共 {len(sem_chunks)} 个chunk\n")
for i, ch in enumerate(sem_chunks[:3]):
    print(f"--- Chunk {i+1} (长度={len(ch)}字符) ---")
    print(ch[:200] + "..." if len(ch) > 200 else ch)
    print()

# 输出相似度分数
scores = sem_splitter.scores
print(f"相邻句子对数: {len(scores)}")
print(f"相似度范围: [{min(scores):.3f}, {max(scores):.3f}]")
print(f"低于阈值的 dips 数量: {sum(1 for s in scores if s < 0.3)}")
print(f"相似度均值: {np.mean(scores):.3f}, 中位数: {np.median(scores):.3f}")

In [ ]:
# 可视化相邻句子相似度
fig, ax = plt.subplots(figsize=(14, 5))

x = range(len(scores))
ax.plot(x, scores, 'b-o', markersize=4, linewidth=1.5, label='Cosine Similarity')
ax.axhline(y=0.3, color='r', linestyle='--', linewidth=1.5, label='Threshold (0.3)')

# 标注 dips
dip_indices = [i for i, s in enumerate(scores) if s < 0.3]
for idx in dip_indices:
    ax.axvline(x=idx, color='orange', alpha=0.3, linewidth=1)

ax.set_xlabel('Adjacent Sentence Pair Index', fontsize=12)
ax.set_ylabel('Cosine Similarity', fontsize=12)
ax.set_title('Semantic Splitter: Adjacent Sentence Similarities (Dips mark chunk boundaries)', fontsize=14)
ax.legend(loc='upper right')
ax.set_ylim(0, 1.05)
plt.tight_layout()
plt.show()

print("橙色竖线标记的是语义'低谷'——这些位置被选为分块边界。")

## 7. 策略5: Adaptive 分块

**核心思想**：根据文档的类型和内容特征，自动选择最合适的分块参数。不同类型的文档有不同的最优分块策略。

**文档类型检测规则**：
- **FAQ / 问答类**：检测到高频的 "Q:" / "A:" / "问：" / "答：" 模式 -> chunk_size=256, overlap=30
- **技术文档**：检测到大量代码块、英文关键词、结构化标题 -> chunk_size=512, overlap=50
- **长报告**：长段落、正式的篇章结构 -> chunk_size=1024, overlap=100
- **默认**：通用文本 -> chunk_size=512, overlap=50

In [ ]:
@dataclass
class AdaptiveSplitter:
    """
    Adaptive splitter that detects document type and chooses chunking parameters.

    Uses sentence-aware splitting internally.
    """
    # 预设配置
    PRESETS = {
        "faq": {"chunk_size": 256, "overlap": 30},
        "technical": {"chunk_size": 512, "overlap": 50},
        "report": {"chunk_size": 1024, "overlap": 100},
        "default": {"chunk_size": 512, "overlap": 50},
    }

    def detect_doc_type(self, text: str) -> str:
        """
        Detect document type based on content patterns.
        Returns one of: "faq" | "technical" | "report" | "default"
        """
        # FAQ 特征：问答模式
        faq_patterns = [
            r'[Qq问]\s*[:：]',
            r'[Aa答]\s*[:：]',
            r'FAQ',
            r'常见问题',
        ]
        faq_count = sum(len(re.findall(p, text)) for p in faq_patterns)
        if faq_count >= 3:
            return "faq"

        # 技术文档特征：代码、函数名、技术关键词
        tech_patterns = [
            r'def\s+\w+\s*\(',  # Python函数
            r'```',               # 代码块
            r'import\s+',         # import语句
            r'class\s+\w+',      # 类定义
            r'function\s+\w+',   # JS函数
        ]
        tech_count = sum(len(re.findall(p, text)) for p in tech_patterns)
        if tech_count >= 2:
            return "technical"

        # 报告特征：长段落、章节标题
        paragraphs = [p for p in text.split('\n\n') if p.strip()]
        avg_para_len = np.mean([len(p) for p in paragraphs]) if paragraphs else 0
        chapter_pattern = r'第[一二三四五六七八九十\d]+[章节篇]'
        chapter_count = len(re.findall(chapter_pattern, text))
        if avg_para_len > 300 or chapter_count >= 3:
            return "report"

        return "default"

    def split(self, text: str) -> Tuple[List[str], str, dict]:
        """
        Split text using adaptive parameters.
        Returns: (chunks, detected_type, params_used)
        """
        doc_type = self.detect_doc_type(text)
        params = self.PRESETS[doc_type]

        # 使用 SentenceSplitter 作为内部分块器
        splitter = SentenceSplitter(
            chunk_size=params["chunk_size"],
            chunk_overlap=params["overlap"],
            language="zh"
        )
        chunks = splitter.split(text)
        return chunks, doc_type, params


print("[OK] AdaptiveSplitter defined.")

In [ ]:
# 创建3种不同类型的测试文本

# 1. FAQ风格
faq_text = """
常见问题解答 FAQ

问：什么是RAG技术？
答：RAG全称为检索增强生成，是一种将信息检索与文本生成相结合的技术。它先从知识库中检索相关文档，然后将检索结果作为上下文输入大语言模型进行生成，有效解决LLM的知识截止和幻觉问题。

问：RAG与传统搜索有什么区别？
答：传统搜索返回的是文档链接或摘要列表，用户需要自行阅读理解。RAG则直接生成基于检索结果的综合答案，用户体验更加自然高效。

问：RAG系统的核心组件有哪些？
答：文档加载器、文本分块器、嵌入模型、向量数据库、检索器和生成器是RAG系统的六大核心组件。每个组件都对系统最终表现有重要影响。

问：如何选择合适的分块策略？
答：对于FAQ类问答文档，推荐使用较小的分块如256字符，因为每个问答对通常较短且独立。对于技术文档，推荐512字符的分块并保持句子完整性。对于长篇报告，可以使用1024字符的大块以保留更多上下文。
"""

# 2. 技术文档风格
tech_text = """
RAG Pipeline 技术实现文档

1. 文档加载模块

```python
class DocumentLoader:
    def __init__(self, file_path: str):
        self.file_path = file_path

    def load_pdf(self) -> str:
        import PyPDF2
        with open(self.file_path, 'rb') as f:
            reader = PyPDF2.PdfReader(f)
            text = ''.join(page.extract_text() for page in reader.pages)
        return text
```

2. 文本分块模块

分块模块负责将长文档切分为合适大小的文本块。核心参数包括chunk_size和chunk_overlap。chunk_size决定了每个块的字符数或Token数，chunk_overlap控制相邻块之间的重叠区域。

3. 嵌入与索引模块

使用OpenAI的text-embedding-3-small模型将文本块转换为1536维向量。向量存储在Qdrant向量数据库中，使用余弦相似度进行最近邻检索。索引建立采用HNSW算法以平衡速度和精度。
"""

# 3. 报告风格
report_text = """
2024年度人工智能技术发展研究报告

第一章 引言

本报告旨在全面回顾2024年度人工智能技术的重大进展。从大语言模型到多模态系统，从自动驾驶到医疗AI，人工智能技术正在以前所未有的速度渗透到各行各业。本报告基于公开论文、产业报告和专家访谈，力求为读者提供一幅完整的技术全景图。

第二章 大语言模型的发展

2024年，大语言模型领域竞争异常激烈。OpenAI发布了GPT-4 Turbo，Google推出了Gemini Ultra，Anthropic发布了Claude 3系列。国产模型如通义千问、文心一言、智谱清言也取得了长足进步。模型能力在推理、数学、编程等多个维度持续提升，同时推理成本和延迟也在不断下降。

第三章 多模态人工智能

多模态AI成为2024年最热门的研究方向之一。GPT-4V、Gemini和Claude 3均具备原生多模态能力，能够理解和生成文本、图像、音频和视频内容。多模态RAG系统能够同时检索和利用多种格式的信息源，大大扩展了AI系统的应用场景。

第四章 结论与展望

2024年的人工智能发展呈现出几个显著趋势：模型能力持续增强，多模态融合加速，AI Agent走向实用，以及AI安全和对齐日益受到重视。展望2025年，我们预期将在通用人工智能AGI的基础理论研究方面取得更多突破。
"""

# 使用自适应分块器
adaptive_splitter = AdaptiveSplitter()

for name, text in [("FAQ", faq_text), ("技术文档", tech_text), ("报告", report_text)]:
    chunks, doc_type, params = adaptive_splitter.split(text)
    print(f"{'='*60}")
    print(f"文档: {name}")
    print(f"检测类型: {doc_type}")
    print(f"使用参数: chunk_size={params['chunk_size']}, overlap={params['overlap']}")
    print(f"生成chunk数: {len(chunks)}")
    print(f"平均chunk大小: {np.mean([len(c) for c in chunks]):.0f} 字符")
    print(f"第一个chunk: {chunks[0][:80]}..." if len(chunks) > 0 else "无")
    print()

## 8. 可视化对比

截取测试文档的前 1500 个字符，可视化展示 5 种策略分别在哪里放置分块边界。

In [ ]:
# 可视化5种策略的边界位置

# 使用前1500字符的文本片段
short_text = test_text[:1500]
print(f"用于可视化的文本长度: {len(short_text)} 字符\n")

# 获取5种策略的边界位置
def get_boundary_positions(text: str, chunks: List[str]) -> List[int]:
    """返回每个chunk在原文中结束位置的字符索引列表"""
    positions = []
    pos = 0
    for ch in chunks:
        pos += len(ch)
        if pos < len(text):
            positions.append(pos)
    return positions

# 为可视化创建各策略的分块结果
f_splitter_viz = FixedLengthSplitter(chunk_size=200, overlap=20)
f_chunks_viz = f_splitter_viz.split(short_text)

r_splitter_viz = RecursiveCharacterSplitter(chunk_size=200, chunk_overlap=20)
r_chunks_viz = r_splitter_viz.split(short_text)

s_splitter_viz = SentenceSplitter(chunk_size=200, chunk_overlap=20, language="zh")
s_chunks_viz = s_splitter_viz.split(short_text)

sem_splitter_viz = SemanticSplitter(chunk_size=200, chunk_overlap=20, similarity_threshold=0.3)
sem_chunks_viz = sem_splitter_viz.split(short_text)

a_splitter_viz = AdaptiveSplitter()
a_chunks_viz, a_type, a_params = a_splitter_viz.split(short_text)

# 获取各策略的边界位置
strategies = {
    "Fixed-Length": get_boundary_positions(short_text, f_chunks_viz),
    "Recursive": get_boundary_positions(short_text, r_chunks_viz),
    "Sentence-Aware": get_boundary_positions(short_text, s_chunks_viz),
    "Semantic": get_boundary_positions(short_text, sem_chunks_viz),
    "Adaptive": get_boundary_positions(short_text, a_chunks_viz),
}

# 绘图
fig, ax = plt.subplots(figsize=(16, 5))

colors = ['#1f77b4', '#ff7f0e', '#2ca02c', '#d62728', '#9467bd']
y_positions = list(range(len(strategies)))

for (name, positions), color, y in zip(strategies.items(), colors, y_positions):
    # 绘制水平条（表示文本范围）
    ax.barh(y, len(short_text), height=0.4, color=color, alpha=0.15, align='center')
    # 标注边界位置
    for pos in positions:
        ax.axvline(x=pos, ymin=(y-0.3)/len(strategies), ymax=(y+0.4)/len(strategies),
                   color=color, linewidth=2, alpha=0.9)
        ax.scatter(pos, y, color=color, s=30, zorder=5)

ax.set_yticks(y_positions)
ax.set_yticklabels(strategies.keys(), fontsize=11)
ax.set_xlabel('Character Position in Text', fontsize=12)
ax.set_title('Chunk Boundary Positions — 5 Strategies Compared (first 1500 chars)', fontsize=14)
ax.set_xlim(0, len(short_text) + 50)

# 图例
legend_patches = [mpatches.Patch(color=c, label=n) for c, n in zip(colors, strategies.keys())]
ax.legend(handles=legend_patches, loc='upper right', fontsize=9)

plt.tight_layout()
plt.show()

print("观察要点：")
print("- Fixed-Length边界均匀分布（机械式）")
print("- Recursive边界倾向于自然分隔符")
print("- Sentence-Aware边界在句子结束处")
print("- Semantic边界在语义发生变化处")
print("- Adaptive根据内容类型自动调整")

## 9. 检索质量对比

模拟检索质量评估：使用简单的关键词重叠作为检索质量的代理指标，比较 5 种策略在实际查询场景中的表现。

In [ ]:
# 检索质量模拟对比

# 构建所有5种策略的完整chunks
all_strategies = {
    "Fixed-Length": FixedLengthSplitter(chunk_size=500, overlap=50).split(test_text),
    "Recursive": RecursiveCharacterSplitter(chunk_size=500, chunk_overlap=50).split(test_text),
    "Sentence-Aware": SentenceSplitter(chunk_size=500, chunk_overlap=50, language="zh").split(test_text),
    "Semantic": SemanticSplitter(chunk_size=500, chunk_overlap=50, similarity_threshold=0.3).split(test_text),
    "Adaptive": AdaptiveSplitter().split(test_text)[0],
}

# 创建5个测试查询
test_queries = [
    "什么是监督学习？",
    "Transformer架构的核心创新是什么？",
    "RAG技术如何解决大语言模型的局限性？",
    "CNN在计算机视觉中有哪些应用？",
    "机器学习的未来发展方向是什么？",
]

# 简单的关键词检索评分函数
def keyword_score(query: str, chunk: str) -> float:
    """
    计算查询关键词在chunk中的覆盖率。
    使用中英文混合提取。
    """
    # 提取查询中的关键词（中文2-gram + 英文词）
    query_terms = set()
    # 中文字符 bigrams
    chinese_chars = re.findall(r'[\u4e00-\u9fff]', query)
    for i in range(len(chinese_chars) - 1):
        query_terms.add(chinese_chars[i] + chinese_chars[i+1])
    # 英文词
    english_words = re.findall(r'[a-zA-Z]+', query.lower())
    query_terms.update(english_words)

    if not query_terms:
        return 0.0

    # 计算覆盖率
    chunk_lower = chunk.lower()
    hit_count = sum(1 for term in query_terms if term in chunk_lower)
    return hit_count / len(query_terms)

# 评估
results = {name: [] for name in all_strategies}

for query in test_queries:
    for strategy_name, chunks in all_strategies.items():
        # 获取该策略下每个chunk与查询的分数
        scores = [keyword_score(query, ch) for ch in chunks]
        # 取最佳匹配分数（模拟 top-1 检索）
        best_score = max(scores) if scores else 0.0
        results[strategy_name].append(best_score)

# 输出表格
print(f"{'Query':<40} ", end="")
for name in all_strategies:
    print(f"{name:<16}", end="")
print()
print("-" * 120)

for i, query in enumerate(test_queries):
    short_q = query[:38] + ".." if len(query) > 38 else query
    print(f"{short_q:<40} ", end="")
    for name in all_strategies:
        score = results[name][i]
        bar = "█" * int(score * 10)
        print(f"{score:.2f} {bar:<10}", end="")
    print()

# 平均分
print("-" * 120)
print(f"{'AVERAGE':<40} ", end="")
avg_scores = {}
for name in all_strategies:
    avg = np.mean(results[name])
    avg_scores[name] = avg
    print(f"{avg:.2f}          ", end="")
print()
print()

In [ ]:
# 可视化检索质量对比
fig, ax = plt.subplots(figsize=(12, 6))

x = np.arange(len(test_queries))
width = 0.15

for idx, (name, scores) in enumerate(results.items()):
    offset = (idx - 2) * width
    bars = ax.bar(x + offset, scores, width, label=name, alpha=0.85)

ax.set_xlabel('Test Query', fontsize=12)
ax.set_ylabel('Best Keyword Overlap Score', fontsize=12)
ax.set_title('Retrieval Quality Comparison: 5 Chunking Strategies x 5 Queries', fontsize=14)
ax.set_xticks(x)
ax.set_xticklabels([f'Q{i+1}' for i in range(len(test_queries))])
ax.legend(loc='upper right')
ax.set_ylim(0, 1.1)

plt.tight_layout()
plt.show()

# 打印查询内容映射
print("\n查询列表：")
for i, q in enumerate(test_queries):
    print(f"  Q{i+1}: {q}")

In [ ]:
# 汇总统计
print("=" * 70)
print("各策略汇总统计")
print("=" * 70)

summary_data = []
for name, chunks in all_strategies.items():
    lens = [len(c) for c in chunks]
    summary_data.append({
        "策略": name,
        "Chunk数": len(chunks),
        "平均长度": f"{np.mean(lens):.0f}",
        "最小长度": min(lens),
        "最大长度": max(lens),
        "标准差": f"{np.std(lens):.0f}",
        "平均检索分": f"{avg_scores[name]:.3f}",
    })

# 简单打印表格
header = f"{'策略':<18} {'Chunks':<8} {'Avg Len':<9} {'Min':<6} {'Max':<6} {'Std':<7} {'检索分':<8}"
print(header)
print("-" * 70)
for d in summary_data:
    print(f"{d['策略']:<18} {d['Chunk数']:<8} {d['平均长度']:<9} {d['最小长度']:<6} {d['最大长度']:<6} {d['标准差']:<7} {d['平均检索分']:<8}")

## 10. 实际建议

### 策略选择指南

| 场景 | 推荐策略 | 推荐参数 | 理由 |
|------|---------|---------|------|
| **快速原型/简单应用** | Fixed-Length | chunk_size=500, overlap=50 | 实现简单，性能高，适合初期开发 |
| **通用自然语言文档** | Sentence-Aware | chunk_size=500, overlap=50 | 保持句子完整，检索质量好 |
| **代码/结构化文档** | Recursive Character | chunk_size=512, overlap=80 | 按分隔符层级切分，保留结构 |
| **高质量语义检索** | Semantic | chunk_size=500, threshold=0.5~0.7 | 语义连贯性好，检索最精准 |
| **多类型文档混合** | Adaptive | 自动检测 | 按内容类型自动适配，开发效率高 |
| **FAQ/问答系统** | Adaptive (FAQ) | chunk_size=256, overlap=30 | 小块匹配问答对的短文本特性 |
| **长报告/论文** | Sentence-Aware | chunk_size=1024, overlap=100 | 大块保留完整段落和上下文 |

### 关键参数调优建议

1. **chunk_size**:
   - 太小（<200）：上下文不足，检索可能遗漏关键信息
   - 太大（>2000）：噪声过多，LLM 上下文窗口浪费
   - **推荐范围**：256-1024 字符（中文） / 128-512 tokens（英文）

2. **chunk_overlap**:
   - 太小：块之间信息断裂，边界附近的信息难以检索
   - 太大：存储冗余，检索重复
   - **推荐范围**：chunk_size 的 10%-20%

3. **语义相似度阈值** (仅 Semantic 策略):
   - 太低：块太大，语义混杂
   - 太高：块太小太碎
   - **推荐范围**：0.5-0.7

### 实践建议

- **先简单后复杂**：从 Fixed-Length 或 Sentence-Aware 开始，根据实际检索质量逐步升级
- **A/B 测试**：对关键场景进行不同策略的检索质量对比
- **监控反馈**：收集用户反馈和检索指标，持续优化分块策略
- **元数据保留**：无论使用哪种策略，都应保留文档的元数据（标题、章节、页码等）

## 11. 总结

本 Notebook 深入实现了 5 种文本分块策略并进行了系统对比：

1. **Fixed-Length 分块** — 最简洁，纯机械切分，适合快速原型
2. **Recursive Character 分块** — 利用分隔符层级，LangChain 的标准方案
3. **Sentence-Aware 分块** — 保证句子完整性，推荐作为生产环境的基线策略
4. **Semantic 分块** — 基于语义相似度，检索质量最高但计算成本较大
5. **Adaptive 分块** — 智能适配文档类型，适合异构文档场景

### 核心要点

- **没有万能的分块策略** — 选择取决于文档类型、检索需求和资源约束
- **句子完整性至关重要** — 在句子中间切分会严重降低检索质量
- **Overlap 是必要的妥协** — 用少量存储冗余换取边界信息的可检索性
- **分块是 RAG 的基础** — 分块质量直接影响后续的检索、生成和用户体验

### 后续延伸

- 探索基于文档层次结构（标题、章节）的分块
- 尝试使用 LLM 进行语义分块（LLMChunker）
- 实验不同 embedding 模型对 Semantic 分块的影响
- 将分块策略集成到完整的 RAG pipeline 中进行端到端评估

---

*Notebook 03: 文本分块深度剖析 — RAG 核心概念系列*